# Senior Video Editor — FIXED
Run Cell 1, then Cell 2.


In [ ]:
# ============================================================
# CELL 1 — SETUP
# ============================================================

from google.colab import drive, output
drive.mount('/content/drive')

# Enable ipywidgets inside Colab
output.enable_custom_widget_manager()

!apt-get update -qq
!apt-get install -y -qq ffmpeg
!pip install -q faster-whisper rapidfuzz ipywidgets

import os, re, json, shutil, subprocess
from datetime import datetime

BASE = "/content/drive/MyDrive/SeniorVideoEditor"
PROJECTS = os.path.join(BASE, "projects")
EXPORTS = os.path.join(BASE, "exports")
TEMP = "/content/senior_video_editor_temp"

for folder in [BASE, PROJECTS, EXPORTS, TEMP]:
    os.makedirs(folder, exist_ok=True)

print("✅ Setup complete")
print("📁 Projects:", PROJECTS)
print("🎬 Exports:", EXPORTS)


In [ ]:
# ============================================================
# CELL 2 — COMPLETE SENIOR VIDEO EDITOR WITH LIVE PROGRESS
# Direct Colab UI — No Gradio / No public link
# ============================================================

import os, re, json, shutil, subprocess, time
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output, Video
from faster_whisper import WhisperModel
from rapidfuzz.fuzz import ratio

print("⏳ Loading speech model...")
whisper_model = WhisperModel("base", device="cpu", compute_type="int8")
print("✅ Speech model ready")

def natural_key(path):
    name = os.path.basename(path)
    return [int(x) if x.isdigit() else x.lower() for x in re.split(r'(\d+)', name)]

def normalize_word(word):
    return re.sub(r"[^a-zA-Z0-9']", "", str(word).lower())

def split_sentences(text):
    text = re.sub(r'\s+', ' ', text.strip())
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]

def get_audio_duration(path):
    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    return float(result.stdout.strip())

def word_matches(a, b):
    if a == b:
        return True
    if len(a) <= 2 or len(b) <= 2:
        return False
    return ratio(a, b) >= 82

def transcribe_words(audio_path, progress_cb=None):
    audio_duration = get_audio_duration(audio_path)

    segments, _ = whisper_model.transcribe(
        audio_path,
        word_timestamps=True,
        vad_filter=True,
        beam_size=1
    )

    words = []
    last_pct = -1

    for seg in segments:
        if seg.words:
            for w in seg.words:
                clean = normalize_word(w.word)
                if clean:
                    words.append({
                        "word": clean,
                        "start": float(w.start),
                        "end": float(w.end)
                    })

        if progress_cb:
            pct = int(min(100, max(0, (float(seg.end) / max(audio_duration, 0.001)) * 100)))
            if pct != last_pct:
                progress_cb(pct, f"🎙 Audio analysis: {pct}%  ({float(seg.end):.1f}s / {audio_duration:.1f}s)")
                last_pct = pct

    if progress_cb:
        progress_cb(100, "✅ Audio analysis complete")

    return words

def build_sentence_timings(audio_path, script_text, progress_cb=None):
    sentences = split_sentences(script_text)
    words = transcribe_words(audio_path, progress_cb=progress_cb)
    audio_duration = get_audio_duration(audio_path)

    if not sentences:
        raise Exception("Script me sentences nahi mile.")
    if not words:
        raise Exception("Audio transcription nahi ban saki.")

    timings = []
    current_word = 0
    total = len(sentences)

    for scene_number, sentence in enumerate(sentences, start=1):
        targets = [normalize_word(w) for w in re.findall(r"[A-Za-z0-9']+", sentence)]
        targets = [w for w in targets if w]
        if not targets:
            continue

        first_targets = targets[:min(5, len(targets))]
        last_targets = targets[-min(5, len(targets)):]

        best_start = current_word
        best_start_score = -1
        search_limit = min(len(words), current_word + max(80, len(targets) * 4))

        for candidate in range(current_word, search_limit):
            score = 0
            for offset, target in enumerate(first_targets):
                wi = candidate + offset
                if wi >= len(words):
                    break
                if word_matches(words[wi]["word"], target):
                    score += 1
            if score > best_start_score:
                best_start_score = score
                best_start = candidate

        expected_end = min(len(words) - 1, best_start + len(targets) + 15)
        best_end = expected_end
        best_end_score = -1

        for candidate in range(max(best_start, expected_end - 25),
                               min(len(words), expected_end + 30)):
            start_compare = max(0, candidate - len(last_targets) + 1)
            actual_words = [w["word"] for w in words[start_compare:candidate + 1]]
            score = sum(
                1 for target in last_targets
                if any(word_matches(a, target) for a in actual_words)
            )
            if score > best_end_score:
                best_end_score = score
                best_end = candidate

        start_time = words[best_start]["start"]
        end_time = words[best_end]["end"]

        if timings:
            start_time = max(start_time, timings[-1]["end"])

        end_time = max(end_time, start_time + 0.25)

        timings.append({
            "scene": scene_number,
            "sentence": sentence,
            "start": round(start_time, 3),
            "end": round(end_time, 3),
            "duration": round(end_time - start_time, 3)
        })

        current_word = min(len(words) - 1, best_end + 1)

        if progress_cb:
            pct = int((scene_number / max(total, 1)) * 100)
            progress_cb(pct, f"📝 Sentence timing: {scene_number}/{total}")

    if timings:
        timings[0]["start"] = 0.0

        for i in range(len(timings) - 1):
            next_start = timings[i + 1]["start"]
            if next_start > timings[i]["start"]:
                timings[i]["end"] = next_start
                timings[i]["duration"] = round(
                    timings[i]["end"] - timings[i]["start"], 3
                )

        timings[-1]["end"] = round(audio_duration, 3)
        timings[-1]["duration"] = round(
            max(0.25, audio_duration - timings[-1]["start"]), 3
        )

    return timings

def save_project(audio_file, image_files, script_text, progress_cb=None):
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    project_dir = os.path.join(PROJECTS, f"project_{stamp}")
    audio_dir = os.path.join(project_dir, "audio")
    image_dir = os.path.join(project_dir, "images")
    os.makedirs(audio_dir, exist_ok=True)
    os.makedirs(image_dir, exist_ok=True)

    if progress_cb:
        progress_cb(2, "📁 Creating project folders...")

    audio_ext = os.path.splitext(audio_file)[1] or ".mp3"
    audio_path = os.path.join(audio_dir, "narration" + audio_ext)
    shutil.copy(audio_file, audio_path)

    if progress_cb:
        progress_cb(5, "🎙 Audio copied")

    saved_images = []
    total_images = len(image_files)
    for i, img in enumerate(image_files, start=1):
        ext = os.path.splitext(img)[1] or ".jpg"
        dst = os.path.join(image_dir, f"{i:04d}{ext.lower()}")
        shutil.copy(img, dst)
        saved_images.append(dst)

        if progress_cb and (i == 1 or i == total_images or i % 10 == 0):
            pct = 5 + int((i / max(total_images, 1)) * 10)
            progress_cb(pct, f"🖼 Copying images: {i}/{total_images}")

    with open(os.path.join(project_dir, "script.txt"), "w", encoding="utf-8") as f:
        f.write(script_text)

    if progress_cb:
        progress_cb(16, "📝 Script saved")
        progress_cb(18, "🎙 Starting audio analysis...")

    # Map audio-analysis 0-100 into overall project progress 18-80
    def timing_progress(inner_pct, message):
        if progress_cb:
            overall = 18 + int((inner_pct / 100) * 62)
            progress_cb(overall, message)

    timings = build_sentence_timings(audio_path, script_text, progress_cb=timing_progress)

    if progress_cb:
        progress_cb(90, "💾 Saving sentence timing file...")

    with open(os.path.join(project_dir, "sentence_timings.json"),
              "w", encoding="utf-8") as f:
        json.dump(timings, f, indent=2, ensure_ascii=False)

    if progress_cb:
        progress_cb(100, "✅ PROJECT READY")

    return project_dir, timings, len(saved_images)

def render_final_video(project_dir, progress_cb=None):
    audio_dir = os.path.join(project_dir, "audio")
    image_dir = os.path.join(project_dir, "images")
    timing_path = os.path.join(project_dir, "sentence_timings.json")

    audio_files = [
        os.path.join(audio_dir, f)
        for f in os.listdir(audio_dir)
        if f.lower().endswith((".mp3", ".wav", ".m4a", ".aac", ".flac", ".ogg"))
    ]
    if not audio_files:
        raise Exception("Narration audio nahi mili.")

    audio_path = audio_files[0]

    images = sorted([
        os.path.join(image_dir, f)
        for f in os.listdir(image_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
    ], key=natural_key)

    if not images:
        raise Exception("Images nahi mili.")

    with open(timing_path, "r", encoding="utf-8") as f:
        timings = json.load(f)

    audio_duration = get_audio_duration(audio_path)
    usable = min(len(images), len(timings))
    if usable == 0:
        raise Exception("Export ke liye scenes nahi hain.")

    if progress_cb:
        progress_cb(5, "🧩 Building exact image timeline...")

    concat_file = os.path.join(TEMP, "final_timeline.txt")
    with open(concat_file, "w", encoding="utf-8") as f:
        for i in range(usable):
            img = images[i]
            duration = max(0.15, float(timings[i]["duration"]))
            safe = img.replace("'", "'\\''")
            f.write(f"file '{safe}'\n")
            f.write(f"duration {duration:.6f}\n")

        last_image = images[usable - 1].replace("'", "'\\''")
        f.write(f"file '{last_image}'\n")

    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = os.path.join(EXPORTS, f"SeniorVideo_FINAL_{stamp}.mp4")

    vf = (
        "scale=1920:1080:force_original_aspect_ratio=increase,"
        "crop=1920:1080,"
        "fps=30,"
        "format=yuv420p"
    )

    cmd = [
        "ffmpeg", "-y",
        "-f", "concat", "-safe", "0", "-i", concat_file,
        "-i", audio_path,
        "-map", "0:v:0", "-map", "1:a:0",
        "-vf", vf,
        "-c:v", "libx264",
        "-preset", "medium",
        "-crf", "20",
        "-pix_fmt", "yuv420p",
        "-c:a", "aac",
        "-b:a", "192k",
        "-ar", "48000",
        "-ac", "2",
        "-t", f"{audio_duration:.6f}",
        "-movflags", "+faststart",
        "-progress", "pipe:1",
        "-nostats",
        output_path
    ]

    if progress_cb:
        progress_cb(10, "🎬 FFmpeg render started...")

    p = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1
    )

    last_pct = -1

    while True:
        line = p.stdout.readline()
        if not line and p.poll() is not None:
            break

        if line.startswith("out_time_ms="):
            try:
                us = int(line.strip().split("=", 1)[1])
                seconds = us / 1_000_000
                pct = int(min(100, max(0, (seconds / max(audio_duration, 0.001)) * 100)))

                if pct != last_pct:
                    overall = 10 + int((pct / 100) * 88)
                    progress_cb(overall, f"🎬 Rendering: {pct}%  ({seconds:.1f}s / {audio_duration:.1f}s)")
                    last_pct = pct
            except:
                pass

    stderr = p.stderr.read()
    code = p.wait()

    if code != 0:
        raise Exception("FFmpeg failed:\n" + stderr[-4000:])

    final_duration = get_audio_duration(output_path)

    if progress_cb:
        progress_cb(100, "✅ FINAL VIDEO COMPLETE")

    return {
        "output_path": output_path,
        "audio_duration": audio_duration,
        "video_duration": final_duration,
        "difference": abs(audio_duration - final_duration),
        "images_used": usable,
        "sentences": len(timings)
    }

def uploaded_items(uploader):
    value = uploader.value
    if not value:
        return []
    if isinstance(value, (tuple, list)):
        return list(value)
    if isinstance(value, dict):
        items = []
        for name, data in value.items():
            if isinstance(data, dict):
                item = dict(data)
                item.setdefault("name", name)
                items.append(item)
        return items
    return []

def save_uploaded_item(item, path):
    content = item.get("content")
    if hasattr(content, "tobytes"):
        content = content.tobytes()
    with open(path, "wb") as f:
        f.write(content)

# -------------------------
# UI
# -------------------------

title = widgets.HTML("""
<h2>🎬 Senior Video Editor</h2>
<b>Exact Sentence → Image → Narration Sync</b><br>
1 sentence = 1 image<br>
Final MP4 saves directly to Google Drive.
<hr>
""")

audio_upload = widgets.FileUpload(
    accept=".mp3,.wav,.m4a,.aac,.flac,.ogg",
    multiple=False,
    description="Upload Audio"
)

image_upload = widgets.FileUpload(
    accept="image/*",
    multiple=True,
    description="Upload Images"
)

script_box = widgets.Textarea(
    placeholder="Paste exact narration script here...",
    description="Script:",
    layout=widgets.Layout(width="100%", height="250px")
)

create_button = widgets.Button(
    description="1️⃣ CREATE PROJECT + AUTO SYNC",
    button_style="success",
    layout=widgets.Layout(width="330px", height="45px")
)

export_button = widgets.Button(
    description="2️⃣ EXPORT FINAL MP4",
    button_style="primary",
    disabled=True,
    layout=widgets.Layout(width="330px", height="45px")
)

project_progress = widgets.IntProgress(
    value=0, min=0, max=100,
    description="Sync:",
    bar_style="info",
    layout=widgets.Layout(width="70%")
)

project_progress_label = widgets.HTML("<b>Ready</b>")

export_progress = widgets.IntProgress(
    value=0, min=0, max=100,
    description="Export:",
    bar_style="info",
    layout=widgets.Layout(width="70%")
)

export_progress_label = widgets.HTML("<b>Waiting</b>")

project_output = widgets.Output()
export_output = widgets.Output()
current_project = {"path": None}

def update_project_progress(pct, message):
    project_progress.value = max(0, min(100, int(pct)))
    project_progress_label.value = f"<b>{message}</b>"

def update_export_progress(pct, message):
    export_progress.value = max(0, min(100, int(pct)))
    export_progress_label.value = f"<b>{message}</b>"

def on_create(_):
    with project_output:
        clear_output()

    create_button.disabled = True
    export_button.disabled = True
    project_progress.value = 0
    project_progress.bar_style = "info"
    update_project_progress(1, "⏳ Starting...")

    try:
        aud = uploaded_items(audio_upload)
        imgs = uploaded_items(image_upload)
        script = (script_box.value or "").strip()

        if not aud:
            raise Exception("Narration audio upload karo.")
        if not imgs:
            raise Exception("Images upload karo.")
        if not script:
            raise Exception("Script paste karo.")

        temp_upload = os.path.join(TEMP, "uploads")
        if os.path.exists(temp_upload):
            shutil.rmtree(temp_upload)
        os.makedirs(temp_upload, exist_ok=True)

        update_project_progress(2, "📥 Reading uploaded audio...")

        audio_name = aud[0].get("name", "narration.mp3")
        audio_ext = os.path.splitext(audio_name)[1] or ".mp3"
        audio_path = os.path.join(temp_upload, "narration" + audio_ext)
        save_uploaded_item(aud[0], audio_path)

        image_paths = []
        total_imgs = len(imgs)

        for i, item in enumerate(imgs, start=1):
            name = item.get("name", f"{i}.jpg")
            ext = os.path.splitext(name)[1] or ".jpg"
            p = os.path.join(temp_upload, f"{i:04d}{ext.lower()}")
            save_uploaded_item(item, p)
            image_paths.append(p)

            if i == 1 or i == total_imgs or i % 10 == 0:
                update_project_progress(
                    3 + int((i / max(total_imgs, 1)) * 7),
                    f"📥 Preparing uploaded images: {i}/{total_imgs}"
                )

        project_dir, timings, image_count = save_project(
            audio_path, image_paths, script,
            progress_cb=update_project_progress
        )

        current_project["path"] = project_dir
        project_progress.bar_style = "success"
        update_project_progress(100, "✅ PROJECT READY")
        export_button.disabled = False

        with project_output:
            print("✅ PROJECT READY")
            print("📁", project_dir)
            print("📝 Sentences:", len(timings))
            print("🖼 Images:", image_count)
            if len(timings) != image_count:
                print("⚠️ Sentence count aur image count same nahi hain.")

    except Exception as e:
        project_progress.bar_style = "danger"
        update_project_progress(project_progress.value, "❌ ERROR")
        with project_output:
            print("❌ ERROR:")
            print(str(e))
    finally:
        create_button.disabled = False

def on_export(_):
    with export_output:
        clear_output()

    export_button.disabled = True
    export_progress.value = 0
    export_progress.bar_style = "info"
    update_export_progress(1, "⏳ Starting export...")

    try:
        project_dir = current_project.get("path")
        if not project_dir:
            raise Exception("Pehle project create karo.")

        result = render_final_video(
            project_dir,
            progress_cb=update_export_progress
        )

        export_progress.bar_style = "success"
        update_export_progress(100, "✅ FINAL VIDEO COMPLETE")

        with export_output:
            print("✅ FINAL VIDEO COMPLETE")
            print(f"🎙 Narration: {result['audio_duration']:.3f} sec")
            print(f"🎬 Video: {result['video_duration']:.3f} sec")
            print(f"⏱ Difference: {result['difference']:.3f} sec")
            print(f"📝 Sentences: {result['sentences']}")
            print(f"🖼 Images used: {result['images_used']}")
            print("\n💾 Saved to Google Drive:")
            print(result["output_path"])
            display(Video(result["output_path"], embed=False))

    except Exception as e:
        export_progress.bar_style = "danger"
        update_export_progress(export_progress.value, "❌ EXPORT ERROR")
        with export_output:
            print("❌ EXPORT ERROR:")
            print(str(e))
    finally:
        export_button.disabled = False

create_button.on_click(on_create)
export_button.on_click(on_export)

display(
    title,
    widgets.HTML("<b>🎙 Narration Audio</b>"),
    audio_upload,
    widgets.HTML("<br><b>🖼 Sentence Images</b>"),
    image_upload,
    widgets.HTML("<br><b>📝 Full Script</b>"),
    script_box,
    widgets.HTML("<br>"),
    create_button,
    project_progress,
    project_progress_label,
    project_output,
    widgets.HTML("<hr>"),
    export_button,
    export_progress,
    export_progress_label,
    export_output
)

print("✅ UI loaded — progress bars enabled.")
